In [2]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


def vikor(X: np.ndarray, weights: np.ndarray, benefit: np.ndarray, v: float = 0.5):
    X = np.array(X, dtype=float)
    weights = weights / weights.sum()
    m, n = X.shape

    f_best = np.where(benefit, X.max(axis=0), X.min(axis=0))
    f_worst = np.where(benefit, X.min(axis=0), X.max(axis=0))

    S = np.zeros(m)
    R = np.zeros(m)

    for i in range(m):
        diff = weights * (f_best - X[i]) / (f_best - f_worst + 1e-12)
        S[i] = diff.sum()
        R[i] = diff.max()

    S_min, S_max = S.min(), S.max()
    R_min, R_max = R.min(), R.max()

    Q = v * (S - S_min) / (S_max - S_min + 1e-12) + \
        (1 - v) * (R - R_min) / (R_max - R_min + 1e-12)

    ranking = np.argsort(Q)
    return Q, ranking


def topsis(X: np.ndarray, weights: np.ndarray, benefit: np.ndarray):
    X = np.array(X, dtype=float)
    weights = weights / np.sum(weights)
    norm_X = X / np.sqrt((X ** 2).sum(axis=0))
    weighted_X = norm_X * weights

    ideal_best = np.where(benefit, weighted_X.max(axis=0), weighted_X.min(axis=0))
    ideal_worst = np.where(benefit, weighted_X.min(axis=0), weighted_X.max(axis=0))

    dist_best = np.sqrt(((weighted_X - ideal_best) ** 2).sum(axis=1))
    dist_worst = np.sqrt(((weighted_X - ideal_worst) ** 2).sum(axis=1))

    scores = dist_worst / (dist_best + dist_worst)
    ranking = np.argsort(-scores)
    return scores, ranking


def local_sensitivity_weights(X, weights, benefit, method="topsis", v=0.5,
                              perturb=[-0.1, -0.05, 0.05, 0.1], topk=3):

    weights = weights / weights.sum()
    m, n = X.shape

    if method == "topsis":
        base_scores, base_rank = topsis(X, weights, benefit)
    elif method == "vikor":
        base_scores, base_rank = vikor(X, weights, benefit, v)
    else:
        raise ValueError("method must be topsis or vikor")

    base_rank_order = np.argsort(base_rank)

    results = []


    for j in range(n):
        for p in perturb:
            w_new = weights.copy()
            w_new[j] *= (1 + p)
            w_new = w_new / w_new.sum()

            if method == "topsis":
                scores, r = topsis(X, w_new, benefit)
            else:
                scores, r = vikor(X, w_new, benefit, v)

            rank_order = np.argsort(r)

            rho, _ = spearmanr(base_rank_order, rank_order)

            abs_diff = np.abs(rank_order - base_rank_order)
            mean_shift = abs_diff.mean()
            max_shift = abs_diff.max()

            topk_base = set(base_rank_order[:topk])
            topk_new = set(rank_order[:topk])
            topk_overlap = len(topk_base & topk_new) / topk

            results.append({
                "weight_index": j,
                "perturbation": p,
                "spearman_rho": rho,
                "mean_rank_shift": mean_shift,
                "max_rank_shift": max_shift,
                "topk_overlap": topk_overlap
            })

    return pd.DataFrame(results)



if __name__ == "__main__":
    X = np.array([[np.float64(0.9762959721488693), np.float64(-17.2466), np.float64(0.9709479065249562)], [np.float64(0.017814186887043837), np.float64(-16.357599999999998), np.float64(0.90649452350998)], [np.float64(0.029017293827856616), np.float64(-14.401800000000001), np.float64(0.8449086418777338)], [np.float64(0.9770068597723778), np.float64(-13.5128), np.float64(0.9791296773793348)], [np.float64(0.6035920134818468), np.float64(-12.0904), np.float64(0.7915855055913192)], [np.float64(0.9773707999669627), np.float64(-13.690600000000002), np.float64(0.9708680861206632)], [np.float64(0.9819004668617325), np.float64(-17.0688), np.float64(0.9684508702410912)], [np.float64(0.9812519862030169), np.float64(-13.832840000000001), np.float64(0.978408954940134)], [np.float64(0.9809709880319621), np.float64(-12.446000000000002), np.float64(0.2958435186389383)]])
    weights = np.array([0.34905985, 0.33735713, 0.31358302])
    benefit = np.array([True, True, True])

    print("=== TOPSIS ===")
    df_t = local_sensitivity_weights(X, weights, benefit, method="topsis")
    print(df_t)

    print("\n=== VIKOR ===")
    df_v = local_sensitivity_weights(X, weights, benefit, method="vikor", v=0.7)
    print(df_v)


=== TOPSIS ===
    weight_index  perturbation  spearman_rho  mean_rank_shift  max_rank_shift  \
0              0         -0.10      1.000000         0.000000               0   
1              0         -0.05      1.000000         0.000000               0   
2              0          0.05      1.000000         0.000000               0   
3              0          0.10      0.983333         0.222222               1   
4              1         -0.10      1.000000         0.000000               0   
5              1         -0.05      1.000000         0.000000               0   
6              1          0.05      1.000000         0.000000               0   
7              1          0.10      1.000000         0.000000               0   
8              2         -0.10      0.983333         0.222222               1   
9              2         -0.05      1.000000         0.000000               0   
10             2          0.05      1.000000         0.000000               0   
11           